<a href="https://colab.research.google.com/github/aarshitaacharya/peft-techniques/blob/main/1_Lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers peft datasets

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict
from datasets import load_dataset


In [ ]:
from datasets import load_dataset

# Let us load the dataset:
dataset = load_dataset("THUDM/humaneval-x")


In [ ]:
import random
model_name = "bigscience/bloomz-560m"

# Load the tokenizer for your pre-trained model
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name)

def tokenize_function(examples):
    tokenized_inputs = tokenizer(examples['prompt'], padding="max_length", truncation=True, max_length=256)

    tokenized_inputs['labels'] = tokenized_inputs['input_ids'].copy()

    return tokenized_inputs


# Tokenize the dataset with batched tokenization
tokenized_dataset = dataset['test'].map(tokenize_function, batched=True)

# Check the columns after tokenization
print("Columns after tokenization:", tokenized_dataset.column_names)

# Check for mismatched lengths
for i in range(len(tokenized_dataset)):
    input_len = len(tokenized_dataset[i]['input_ids'])
    label_len = len(tokenized_dataset[i]['labels'])
    assert input_len == label_len, f"Mismatch at index {i}: input_len={input_len}, label_len={label_len}"

train_size = int(0.8 * len(tokenized_dataset))
train_indices = random.sample(range(len(tokenized_dataset)), train_size)
train_dataset = tokenized_dataset.select(train_indices)

# Create evaluation dataset from remaining indices
eval_indices = list(set(range(len(tokenized_dataset))) - set(train_indices))
eval_dataset = tokenized_dataset.select(eval_indices)

# Format dataset for PyTorch
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
eval_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

train_dataset = train_dataset.with_format("torch", columns=['input_ids', 'attention_mask', 'labels'])
eval_dataset = eval_dataset.with_format("torch", columns=['input_ids', 'attention_mask', 'labels'])

Columns after tokenization: ['task_id', 'prompt', 'declaration', 'canonical_solution', 'test', 'example_test', 'input_ids', 'attention_mask', 'labels']


In [ ]:
print(dataset['test'][1])

{'task_id': 'Python/1', 'prompt': 'from typing import List\n\n\ndef separate_paren_groups(paren_string: str) -> List[str]:\n    """ Input to this function is a string containing multiple groups of nested parentheses. Your goal is to\n    separate those group into separate strings and return the list of those.\n    Separate groups are balanced (each open brace is properly closed) and not nested within each other\n    Ignore any spaces in the input string.\n    >>> separate_paren_groups(\'( ) (( )) (( )( ))\')\n    [\'()\', \'(())\', \'(()())\']\n    """\n', 'declaration': 'from typing import List\n\n\ndef separate_paren_groups(paren_string: str) -> List[str]:\n', 'canonical_solution': "    result = []\n    current_string = []\n    current_depth = 0\n\n    for c in paren_string:\n        if c == '(':\n            current_depth += 1\n            current_string.append(c)\n        elif c == ')':\n            current_depth -= 1\n            current_string.append(c)\n\n            if current_

In [ ]:
# Define LoRA configuration
lora_config = LoraConfig(
    r=4,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query_key_value"],
)

# Apply LoRA to the model
lora_model = get_peft_model(model, lora_config)


In [22]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq, EarlyStoppingCallback

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = TrainingArguments(
    output_dir="./lora_model",
    evaluation_strategy="steps",
    eval_steps=500,
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    weight_decay=0.01,
    fp16=True,
    save_steps=500,
    save_total_limit=2,
    dataloader_pin_memory=False,
    gradient_accumulation_steps=4,
    load_best_model_at_end=True
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

# Start fine-tuning
trainer.train()


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1525: FutureWarning:

`evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead

/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:494: FutureWarning:

`torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.



Step,Training Loss,Validation Loss


TrainOutput(global_step=40, training_loss=43.829254150390625, metrics={'train_runtime': 103.1821, 'train_samples_per_second': 6.348, 'train_steps_per_second': 0.388, 'total_flos': 295712952680448.0, 'train_loss': 43.829254150390625, 'epoch': 4.848484848484849})

In [20]:
!pip install jupyter-dash
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from jupyter_dash import JupyterDash
import dash_core_components as dcc
import dash_html_components as html
from dash.dependencies import Input, Output
import dash


model.eval()

# Build App
app = dash.Dash(__name__)

app.layout = html.Div(style={'backgroundColor': '#f8f9fa', 'padding': '20px'}, children=[
    html.H1("Model Inference Web App", style={'textAlign': 'center', 'color': '#343a40'}),
    dcc.Input(id='input-text', type='text', placeholder='Enter input text here...',
               style={'width': '100%', 'padding': '10px', 'fontSize': '18px', 'marginBottom': '10px'}),
    html.Button('Submit', id='submit-button', n_clicks=0,
                 style={'backgroundColor': '#007bff', 'color': 'white', 'padding': '10px 20px',
                        'border': 'none', 'borderRadius': '5px', 'cursor': 'pointer'}),
    html.Div(id='output-prediction', style={'marginTop': '20px', 'fontSize': '18px',
                                             'color': '#495057', 'border': '1px solid #ced4da',
                                             'padding': '10px', 'borderRadius': '5px'})
])

# Define callback for model inference
@app.callback(
    Output('output-prediction', 'children'),
    [Input('submit-button', 'n_clicks')],
    [Input('input-text', 'value')]
)
def update_output(n_clicks, input_text):
    if n_clicks > 0 and input_text:
        # Tokenize input text
        inputs = tokenizer(input_text, return_tensors='pt').to(model.device)

        # Perform inference
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1)

        # Decode predictions
        decoded_prediction = tokenizer.decode(predictions[0], skip_special_tokens=True)

        return f"Prediction: {decoded_prediction}"

    return "Enter text and click Submit."

app.run_server()

<IPython.core.display.Javascript object>

In [21]:
!pip install pyngrok
!ngrok authtoken 2nfGHPST8eqZs7wQpzzzFx9E6nV_7YoWjiP1jzEsZ83KypzDB

from pyngrok import ngrok
# Start ngrok tunnel for port 8050 using http
public_url = ngrok.connect(8050, "http")

# Display the public URL
print(public_url)

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
NgrokTunnel: "https://0106-35-197-37-248.ngrok-free.app" -> "http://localhost:8050"
